# EDA TOÀN DIỆN - 54 Features
## Oil Price Prediction Project

**Dataset:** `dataset_step4_transformed.csv` (54 cột, 2923 dòng)  
**Target:** `oil_return` (% thay đổi giá dầu Brent hàng ngày)  
**Train/Test split:** Train < 2023-01-01 | Test >= 2023-01-01

---

### Mục lục
1. Setup & Load Data
2. Tổng quan dữ liệu (shape, dtypes, missing, describe)
3. Phân tích Target Variable (`oil_return`)
4. Phân phối từng nhóm Features
5. Kiểm định tính dừng (ADF + KPSS)
6. ACF/PACF & Volatility Clustering
7. Correlation Analysis (Pearson + Spearman)
8. Multicollinearity (VIF)
9. Outlier Detection (IQR + Z-score)
10. Feature Importance (Mutual Information)
11. Train vs Test Distribution Shift
12. Tổng kết & Khuyến nghị

---
## 1. Setup & Load Data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 100,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
    'figure.facecolor': 'white'
})
sns.set_style('whitegrid')

print('Setup hoàn tất.')

In [ ]:
# Load data
df = pd.read_csv('../data/processed/dataset_step4_transformed.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

# Train/Test split theo thời gian
SPLIT_DATE = '2023-01-01'
train = df[df['date'] < SPLIT_DATE].copy()
test  = df[df['date'] >= SPLIT_DATE].copy()

TARGET = 'oil_return'

# Phân nhóm features
FEATURE_GROUPS = {
    'Market Prices': ['oil_close', 'usd_close', 'sp500_close', 'vix_close'],
    'FRED Macro': ['yield_spread', 'wti_fred', 'fed_funds_rate_lag', 'cpi_lag', 'unemployment_lag'],
    'EIA Supply': ['crude_inventory_weekly', 'crude_production_weekly', 'net_imports_weekly', 'inventory_change_pct'],
    'GDELT Sentiment': ['gdelt_tone', 'gdelt_goldstein', 'gdelt_volume', 'gdelt_events',
                        'gdelt_tone_7d', 'gdelt_tone_30d', 'gdelt_tone_spike',
                        'gdelt_volume_7d', 'gdelt_goldstein_7d'],
    'ACLED Conflict': ['conflict_event_count', 'fatalities'],
    'Auxiliary': ['gdelt_data_imputed'],
    'Market Returns': ['oil_return', 'usd_return', 'sp500_return', 'vix_return'],
    'Volatility': ['oil_volatility_7d'],
    'FRED Derived': ['fed_rate_change', 'recession_signal', 'cpi_yoy', 'fed_rate_regime', 'real_rate'],
    'EIA Derived': ['inventory_zscore', 'production_change_pct', 'net_imports_change_pct'],
    'GDELT Derived': ['gdelt_volume_log', 'media_attention_spike'],
    'ACLED Derived': ['conflict_intensity_7d', 'fatalities_7d'],
    'Stress Index': ['stress_tone', 'stress_volume', 'stress_goldstein', 'geopolitical_stress_index'],
    'Lag Features': ['oil_return_lag1', 'oil_return_lag2', 'vix_lag1', 'gdelt_tone_lag1', 'gdelt_volume_lag1'],
    'Time Features': ['day_of_week', 'month']
}

# Tất cả features (trừ date và target)
ALL_NUMERIC = [c for c in df.columns if c not in ['date', TARGET]]
BINARY_COLS = ['gdelt_tone_spike', 'recession_signal', 'media_attention_spike', 'gdelt_data_imputed']
CONTINUOUS_COLS = [c for c in ALL_NUMERIC if c not in BINARY_COLS]

print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Train: {len(train)} rows ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Test:  {len(test)} rows ({test["date"].min().date()} → {test["date"].max().date()})')
print(f'Target: {TARGET}')
print(f'Features: {len(ALL_NUMERIC)} numeric + 1 date')

---
## 2. Tổng quan dữ liệu

In [ ]:
# 2.1 Data types
print('=== DATA TYPES ===')
dtype_summary = df.dtypes.value_counts()
print(dtype_summary)
print()

# 2.2 Missing values
missing = df.isnull().sum()
print(f'=== MISSING VALUES ===')
print(f'Tổng cột có missing: {(missing > 0).sum()} / {len(df.columns)}')
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print('Không có missing values.')
print()

# 2.3 Infinite values
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df[numeric_cols]).sum().sum()
print(f'=== INFINITE VALUES ===')
print(f'Tổng giá trị INF: {inf_count}')
print()

# 2.4 Duplicate rows
dup_count = df.duplicated().sum()
print(f'=== DUPLICATES ===')
print(f'Số dòng trùng lặp: {dup_count}')

In [ ]:
# 2.5 Thống kê mô tả trên TRAIN set
desc = train[ALL_NUMERIC + [TARGET]].describe().T
desc['skew'] = train[ALL_NUMERIC + [TARGET]].skew()
desc['kurtosis'] = train[ALL_NUMERIC + [TARGET]].kurtosis()
desc['iqr'] = desc['75%'] - desc['25%']

# Hiển thị đẹp
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
display(desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skew', 'kurtosis', 'iqr']])

In [ ]:
# 2.6 Zero-variance & near-zero-variance check (trên train)
print('=== ZERO / NEAR-ZERO VARIANCE CHECK (Train) ===')
for col in ALL_NUMERIC:
    n_unique = train[col].nunique()
    top_freq = train[col].value_counts(normalize=True).iloc[0]
    if n_unique <= 5 or top_freq > 0.95:
        print(f'  {col}: {n_unique} unique values, top value chiếm {top_freq:.1%}')

---
## 3. Phân tích Target Variable (`oil_return`)

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# 3.1 Histogram + KDE
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(train[TARGET], bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')
x_kde = np.linspace(train[TARGET].min(), train[TARGET].max(), 300)
kde = stats.gaussian_kde(train[TARGET])
ax1.plot(x_kde, kde(x_kde), 'r-', lw=2, label='KDE')
# So sánh với Normal
mu, sigma = train[TARGET].mean(), train[TARGET].std()
ax1.plot(x_kde, stats.norm.pdf(x_kde, mu, sigma), 'g--', lw=1.5, label=f'Normal(μ={mu:.4f}, σ={sigma:.4f})')
ax1.set_title('Phân phối oil_return (Train)')
ax1.legend(fontsize=8)

# 3.2 Q-Q Plot
ax2 = fig.add_subplot(gs[0, 1])
stats.probplot(train[TARGET], dist='norm', plot=ax2)
ax2.set_title('Q-Q Plot vs Normal')

# 3.3 Boxplot theo năm
ax3 = fig.add_subplot(gs[0, 2])
train_year = train.copy()
train_year['year'] = train_year['date'].dt.year
train_year.boxplot(column=TARGET, by='year', ax=ax3)
ax3.set_title('oil_return theo Năm')
plt.suptitle('')
ax3.set_xlabel('Năm')
ax3.tick_params(axis='x', rotation=45)

# 3.4 Time series
ax4 = fig.add_subplot(gs[1, 0:2])
ax4.plot(train['date'], train[TARGET], alpha=0.7, lw=0.5, color='steelblue')
ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5)
# Highlight extreme events
events = {
    '2020-03': 'COVID crash',
    '2022-02': 'Ukraine war',
}
for date_str, label in events.items():
    ax4.axvline(x=pd.Timestamp(date_str), color='orange', alpha=0.6, linestyle=':')
    ax4.text(pd.Timestamp(date_str), ax4.get_ylim()[1]*0.9, label, fontsize=8, rotation=90, va='top')
ax4.set_title('oil_return theo thời gian (Train)')
ax4.set_ylabel('oil_return')

# 3.5 Thống kê
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
target_stats = {
    'Mean': f"{train[TARGET].mean():.6f}",
    'Median': f"{train[TARGET].median():.6f}",
    'Std': f"{train[TARGET].std():.6f}",
    'Skewness': f"{train[TARGET].skew():.4f}",
    'Kurtosis': f"{train[TARGET].kurtosis():.4f}",
    'Min': f"{train[TARGET].min():.4f}",
    'Max': f"{train[TARGET].max():.4f}",
    'Jarque-Bera p': f"{stats.jarque_bera(train[TARGET])[1]:.2e}",
    'Shapiro-Wilk p': f"{stats.shapiro(train[TARGET].sample(min(5000, len(train)), random_state=42))[1]:.2e}",
    '% Positive': f"{(train[TARGET] > 0).mean():.1%}",
    '% Negative': f"{(train[TARGET] < 0).mean():.1%}",
    '% Zero': f"{(train[TARGET] == 0).mean():.1%}",
}
text = '\n'.join([f'{k:>16}: {v}' for k, v in target_stats.items()])
ax5.text(0.1, 0.95, 'Thống kê Target (Train)', fontsize=12, fontweight='bold',
         transform=ax5.transAxes, va='top')
ax5.text(0.1, 0.82, text, fontsize=10, fontfamily='monospace',
         transform=ax5.transAxes, va='top')

plt.tight_layout()
plt.show()

print(f"\nNhận xét:")
print(f"- Kurtosis = {train[TARGET].kurtosis():.2f} >> 0 → phân phối leptokurtic (đuôi nặng), đặc trưng dữ liệu tài chính")
print(f"- Skewness = {train[TARGET].skew():.2f} → hơi lệch trái (crash giảm mạnh hơn rally tăng)")
print(f"- Jarque-Bera & Shapiro-Wilk p << 0.05 → KHÔNG phân phối chuẩn")
print(f"- {(train[TARGET] > 0).mean():.1%} ngày tăng vs {(train[TARGET] < 0).mean():.1%} ngày giảm → gần cân bằng")

---
## 4. Phân phối từng nhóm Features

In [ ]:
# 4.1 Histogram grid cho từng nhóm feature
groups_to_plot = [
    ('Market Prices', FEATURE_GROUPS['Market Prices']),
    ('Market Returns', FEATURE_GROUPS['Market Returns']),
    ('FRED Macro', FEATURE_GROUPS['FRED Macro']),
    ('FRED Derived', FEATURE_GROUPS['FRED Derived']),
    ('EIA Supply', FEATURE_GROUPS['EIA Supply']),
    ('EIA Derived', FEATURE_GROUPS['EIA Derived']),
    ('GDELT Sentiment', FEATURE_GROUPS['GDELT Sentiment']),
    ('GDELT Derived + Stress', FEATURE_GROUPS['GDELT Derived'] + FEATURE_GROUPS['Stress Index']),
    ('ACLED + Conflict', FEATURE_GROUPS['ACLED Conflict'] + FEATURE_GROUPS['ACLED Derived']),
    ('Lag + Time + Vol', FEATURE_GROUPS['Lag Features'] + FEATURE_GROUPS['Time Features'] + FEATURE_GROUPS['Volatility']),
]

for group_name, cols in groups_to_plot:
    n = len(cols)
    ncols_plot = min(4, n)
    nrows_plot = (n + ncols_plot - 1) // ncols_plot
    fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(4*ncols_plot, 3*nrows_plot))
    fig.suptitle(f'Phân phối nhóm: {group_name} (Train)', fontsize=14, fontweight='bold', y=1.02)
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    
    for i, col in enumerate(cols):
        ax = axes_flat[i]
        data = train[col].dropna()
        if col in BINARY_COLS:
            data.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
            ax.set_title(f'{col} (binary)')
        else:
            ax.hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='white', density=True)
            try:
                kde = stats.gaussian_kde(data)
                x = np.linspace(data.min(), data.max(), 200)
                ax.plot(x, kde(x), 'r-', lw=1.5)
            except Exception:
                pass
            ax.set_title(f'{col}\nskew={data.skew():.2f} kurt={data.kurtosis():.2f}', fontsize=9)
    
    for j in range(i+1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 4.2 Bảng tổng hợp Skewness & Kurtosis (Train)
sk_table = pd.DataFrame({
    'skewness': train[CONTINUOUS_COLS].skew(),
    'kurtosis': train[CONTINUOUS_COLS].kurtosis(),
    'abs_skew': train[CONTINUOUS_COLS].skew().abs()
}).sort_values('abs_skew', ascending=False)

print('=== TOP 15 FEATURES LỆCH NHẤT (|skewness| cao nhất) ===')
display(sk_table.head(15)[['skewness', 'kurtosis']])

# Flag
highly_skewed = sk_table[sk_table['abs_skew'] > 2].index.tolist()
print(f'\nSố features có |skewness| > 2 (cần xem xét transform): {len(highly_skewed)}')
if highly_skewed:
    print(f'  → {highly_skewed}')

---
## 5. Kiểm định tính dừng (Stationarity Tests)

In [ ]:
# ADF + KPSS cho các features quan trọng
# ADF: H0 = có unit root (non-stationary), reject nếu p < 0.05 → stationary
# KPSS: H0 = stationary, reject nếu p < 0.05 → non-stationary

stationarity_cols = [
    'oil_close', 'oil_return',
    'usd_close', 'usd_return',
    'sp500_close', 'sp500_return',
    'vix_close', 'vix_return',
    'fed_funds_rate_lag', 'fed_rate_change',
    'cpi_lag', 'cpi_yoy',
    'crude_inventory_weekly', 'inventory_zscore',
    'gdelt_tone', 'gdelt_tone_7d',
    'gdelt_volume', 'gdelt_volume_log',
    'geopolitical_stress_index',
    'oil_volatility_7d',
    'real_rate', 'yield_spread',
]

results = []
for col in stationarity_cols:
    series = train[col].dropna()
    # ADF
    adf_stat, adf_p, _, _, _, _ = adfuller(series, autolag='AIC')
    # KPSS
    kpss_stat, kpss_p, _, _ = kpss(series, regression='c', nlags='auto')
    
    if adf_p < 0.05 and kpss_p >= 0.05:
        verdict = 'Stationary'
    elif adf_p >= 0.05 and kpss_p < 0.05:
        verdict = 'Non-stationary'
    elif adf_p < 0.05 and kpss_p < 0.05:
        verdict = 'Trend-stationary'
    else:
        verdict = 'Uncertain'
    
    results.append({
        'feature': col,
        'ADF_stat': adf_stat,
        'ADF_p': adf_p,
        'KPSS_stat': kpss_stat,
        'KPSS_p': kpss_p,
        'verdict': verdict
    })

stat_df = pd.DataFrame(results)
display(stat_df.style.applymap(
    lambda x: 'background-color: #ffcccc' if x == 'Non-stationary' 
    else ('background-color: #ccffcc' if x == 'Stationary' else ''),
    subset=['verdict']
))

non_stat = stat_df[stat_df['verdict'] == 'Non-stationary']['feature'].tolist()
print(f'\nNon-stationary features: {non_stat}')
print('→ Các biến giá gốc (oil_close, usd_close...) non-stationary là bình thường.')
print('→ Phiên bản return/derived của chúng nên stationary.')

---
## 6. ACF/PACF & Volatility Clustering

In [ ]:
# 6.1 ACF/PACF cho oil_return
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_acf(train[TARGET], lags=40, ax=axes[0, 0], title='ACF - oil_return')
plot_pacf(train[TARGET], lags=40, ax=axes[0, 1], title='PACF - oil_return', method='ywm')

# 6.2 ACF/PACF cho oil_return² (volatility clustering / ARCH effects)
squared_returns = train[TARGET] ** 2
plot_acf(squared_returns, lags=40, ax=axes[1, 0], title='ACF - oil_return² (ARCH effects)')
plot_pacf(squared_returns, lags=40, ax=axes[1, 1], title='PACF - oil_return²', method='ywm')

plt.tight_layout()
plt.show()

# Ljung-Box test cho ARCH effects
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_result = acorr_ljungbox(squared_returns, lags=[10, 20, 30], return_df=True)
print('=== Ljung-Box Test trên oil_return² (kiểm tra ARCH effects) ===')
print(lb_result)
print('\nNếu p-value << 0.05 → có volatility clustering (biến động có xu hướng co cụm)')
print('→ Mô hình nên xem xét sử dụng oil_volatility_7d hoặc GARCH-type features')

In [ ]:
# 6.3 Rolling volatility qua thời gian
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(train['date'], train[TARGET], alpha=0.6, lw=0.5, color='steelblue')
axes[0].set_title('oil_return')
axes[0].set_ylabel('Return')

axes[1].plot(train['date'], train['oil_volatility_7d'], color='crimson', lw=0.8)
axes[1].fill_between(train['date'], 0, train['oil_volatility_7d'], alpha=0.3, color='crimson')
axes[1].set_title('oil_volatility_7d (Rolling 7-day Std of Returns)')
axes[1].set_ylabel('Volatility')

for ax in axes:
    for date_str, label in [('2020-03', 'COVID'), ('2022-02', 'Ukraine')]:
        ax.axvline(x=pd.Timestamp(date_str), color='orange', alpha=0.6, linestyle=':')

plt.tight_layout()
plt.show()
print('→ Volatility clustering rõ ràng: biến động cực cao quanh COVID-19 (2020) và chiến tranh Ukraine (2022)')

---
## 7. Correlation Analysis

In [ ]:
# 7.1 Spearman Correlation với Target (Train)
features_for_corr = [c for c in ALL_NUMERIC if c != TARGET]

spearman_corr = train[features_for_corr].corrwith(train[TARGET], method='spearman')
pearson_corr = train[features_for_corr].corrwith(train[TARGET], method='pearson')

corr_comparison = pd.DataFrame({
    'Spearman': spearman_corr,
    'Pearson': pearson_corr,
    'Abs_Spearman': spearman_corr.abs(),
    'Diff': (spearman_corr - pearson_corr).abs()
}).sort_values('Abs_Spearman', ascending=False)

print('=== CORRELATION VỚI TARGET (oil_return) - Train set ===')
display(corr_comparison[['Spearman', 'Pearson', 'Diff']].head(20))

print(f'\nTop 5 Spearman: {corr_comparison.head(5).index.tolist()}')
print(f'Features có |Spearman| > 0.1: {corr_comparison[corr_comparison["Abs_Spearman"] > 0.1].index.tolist()}')
print(f'\nNhận xét: Nếu Pearson ≈ Spearman → quan hệ tuyến tính. Nếu khác nhiều → quan hệ phi tuyến.')

In [ ]:
# 7.2 Bar chart Spearman Correlation với Target
top_n = 25
top_corr = corr_comparison.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in top_corr['Spearman']]
ax.barh(range(top_n), top_corr['Spearman'], color=colors, edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_corr.index)
ax.invert_yaxis()
ax.set_xlabel('Spearman Correlation')
ax.set_title(f'Top {top_n} Features - Spearman Correlation với oil_return (Train)')
ax.axvline(x=0, color='black', lw=0.5)
for i, v in enumerate(top_corr['Spearman']):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f'{v:.3f}', va='center',
            ha='left' if v >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 7.3 Full Correlation Heatmap (Spearman) - Features vs Features
# Chọn features quan trọng để heatmap không quá rối
heatmap_cols = [
    # Target
    'oil_return',
    # Returns
    'usd_return', 'sp500_return', 'vix_return',
    # Lag
    'oil_return_lag1', 'oil_return_lag2', 'vix_lag1', 'gdelt_tone_lag1', 'gdelt_volume_lag1',
    # Macro derived
    'yield_spread', 'fed_rate_change', 'real_rate', 'fed_rate_regime', 'cpi_yoy',
    # Supply derived
    'inventory_zscore', 'production_change_pct', 'net_imports_change_pct', 'inventory_change_pct',
    # Sentiment
    'gdelt_tone_7d', 'gdelt_goldstein_7d', 'gdelt_volume_log', 'geopolitical_stress_index',
    # Conflict
    'conflict_intensity_7d', 'fatalities_7d',
    # Volatility + Time
    'oil_volatility_7d', 'day_of_week', 'month',
]

corr_matrix = train[heatmap_cols].corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={'fontsize': 7}, ax=ax)
ax.set_title('Spearman Correlation Matrix - Key Features (Train)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 7.4 Tìm cặp multicollinear (|corr| > 0.80)
full_corr = train[CONTINUOUS_COLS].corr(method='spearman')
high_corr_pairs = []
for i in range(len(full_corr.columns)):
    for j in range(i+1, len(full_corr.columns)):
        if abs(full_corr.iloc[i, j]) > 0.80:
            high_corr_pairs.append({
                'Feature 1': full_corr.columns[i],
                'Feature 2': full_corr.columns[j],
                'Spearman': full_corr.iloc[i, j]
            })

hc_df = pd.DataFrame(high_corr_pairs).sort_values('Spearman', key=abs, ascending=False)
print(f'=== CẶP FEATURES CÓ |SPEARMAN| > 0.80 ({len(hc_df)} cặp) ===')
display(hc_df.reset_index(drop=True))
print('\n→ Các cặp này có đa cộng tuyến cao, nên xem xét giữ 1 trong 2 hoặc dùng PCA/regularization.')

In [ ]:
# 7.5 Rolling 90-day Spearman Correlation với Target
rolling_corr_features = ['sp500_return', 'usd_return', 'vix_return',
                         'oil_return_lag1', 'geopolitical_stress_index',
                         'inventory_zscore', 'oil_volatility_7d', 'real_rate']

fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
axes_flat = axes.flatten()

for i, feat in enumerate(rolling_corr_features):
    ax = axes_flat[i]
    rolling = train[[TARGET, feat]].rolling(90).corr().unstack()[TARGET][feat]
    ax.plot(train['date'], rolling, color='steelblue', lw=0.8)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.fill_between(train['date'], rolling, 0, alpha=0.15, color='steelblue')
    ax.set_title(f'{feat}', fontsize=10)
    ax.set_ylabel('ρ (90d)')
    for date_str, label in [('2020-03', 'COVID'), ('2022-02', 'UA')]:
        ax.axvline(x=pd.Timestamp(date_str), color='orange', alpha=0.5, linestyle=':')

fig.suptitle('Rolling 90-Day Spearman Correlation với oil_return (Train)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print('→ Correlation thay đổi theo thời gian (regime-dependent). Không ổn định → cần model robust.')

---
## 8. Multicollinearity - VIF (Variance Inflation Factor)

In [ ]:
# VIF tính cho các nhóm features riêng (vì VIF full 53 biến sẽ rất chậm và kém ý nghĩa)

vif_groups = {
    'Market (Returns + Vol)': ['usd_return', 'sp500_return', 'vix_return', 'oil_volatility_7d', 'vix_lag1'],
    'FRED Derived': ['yield_spread', 'fed_rate_change', 'cpi_yoy', 'fed_rate_regime', 'real_rate', 'unemployment_lag'],
    'EIA Derived': ['inventory_zscore', 'production_change_pct', 'net_imports_change_pct', 'inventory_change_pct'],
    'GDELT + Stress': ['gdelt_tone_7d', 'gdelt_goldstein_7d', 'gdelt_volume_log',
                       'geopolitical_stress_index', 'gdelt_tone_lag1', 'gdelt_volume_lag1'],
    'Lags + Time': ['oil_return_lag1', 'oil_return_lag2', 'vix_lag1', 'day_of_week', 'month'],
}

print('=== VIF THEO NHÓM FEATURES (Train set) ===')
print('VIF > 10: đa cộng tuyến NGUY HIỂM | VIF > 5: đáng lo ngại | VIF < 5: OK\n')

all_vif_results = []
for group_name, cols in vif_groups.items():
    X = train[cols].dropna()
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=cols)
    
    vif_data = []
    for i, col in enumerate(cols):
        vif_val = variance_inflation_factor(X_scaled.values, i)
        flag = '⚠ NGUY HIỂM' if vif_val > 10 else ('⚡ Đáng lo' if vif_val > 5 else '✓ OK')
        vif_data.append({'Group': group_name, 'Feature': col, 'VIF': round(vif_val, 2), 'Flag': flag})
    
    vif_df = pd.DataFrame(vif_data)
    all_vif_results.append(vif_df)
    print(f'--- {group_name} ---')
    display(vif_df[['Feature', 'VIF', 'Flag']])
    print()

all_vif = pd.concat(all_vif_results, ignore_index=True)
danger_vif = all_vif[all_vif['VIF'] > 10]
print(f'\nTổng features có VIF > 10: {len(danger_vif)}')
if len(danger_vif) > 0:
    print(danger_vif[['Group', 'Feature', 'VIF']].to_string(index=False))

---
## 9. Outlier Detection

In [ ]:
# 9.1 IQR method
outlier_cols = ['oil_return', 'usd_return', 'sp500_return', 'vix_return',
                'oil_volatility_7d', 'gdelt_tone', 'gdelt_volume',
                'inventory_change_pct', 'production_change_pct', 'net_imports_change_pct',
                'fed_rate_change', 'conflict_event_count', 'fatalities']

outlier_results = []
for col in outlier_cols:
    data = train[col]
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outlier_iqr = ((data < lower) | (data > upper)).sum()
    
    # Z-score method
    z = np.abs(stats.zscore(data))
    n_outlier_z3 = (z > 3).sum()
    
    outlier_results.append({
        'Feature': col,
        'IQR_outliers': n_outlier_iqr,
        'IQR_%': f"{n_outlier_iqr/len(data)*100:.1f}%",
        'Zscore3_outliers': n_outlier_z3,
        'Zscore3_%': f"{n_outlier_z3/len(data)*100:.1f}%",
    })

outlier_df = pd.DataFrame(outlier_results)
print('=== OUTLIER DETECTION (Train set) ===')
display(outlier_df)

In [ ]:
# 9.2 Boxplot grid
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes_flat = axes.flatten()

for i, col in enumerate(outlier_cols):
    ax = axes_flat[i]
    ax.boxplot(train[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='steelblue'),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='o', markersize=2, alpha=0.5))
    ax.set_title(col, fontsize=9)

for j in range(len(outlier_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Boxplots - Outlier Detection (Train)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 9.3 Outliers trên time series - khi nào xảy ra?
fig, ax = plt.subplots(figsize=(16, 5))
data = train[TARGET]
Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
is_outlier = (data < lower) | (data > upper)

ax.plot(train['date'], data, alpha=0.5, lw=0.5, color='steelblue', label='oil_return')
ax.scatter(train['date'][is_outlier], data[is_outlier], color='red', s=10, zorder=5, label=f'Outliers ({is_outlier.sum()})')
ax.axhline(y=upper, color='orange', linestyle='--', alpha=0.5, label=f'IQR bounds [{lower:.3f}, {upper:.3f}]')
ax.axhline(y=lower, color='orange', linestyle='--', alpha=0.5)
ax.legend()
ax.set_title('oil_return Outliers trên Time Series (Train)')
plt.tight_layout()
plt.show()

# Top extreme dates
extreme = train.loc[is_outlier, ['date', TARGET]].sort_values(TARGET)
print('=== TOP 5 NGÀY GIẢM MẠNH NHẤT ===')
print(extreme.head(5).to_string(index=False))
print('\n=== TOP 5 NGÀY TĂNG MẠNH NHẤT ===')
print(extreme.tail(5).to_string(index=False))

---
## 10. Feature Importance - Mutual Information

In [ ]:
# Mutual Information - đo mối quan hệ phi tuyến, không giả định gì về dạng quan hệ
features_mi = [c for c in ALL_NUMERIC if c != TARGET]
X_train = train[features_mi].fillna(0)
y_train = train[TARGET]

mi_scores = mutual_info_regression(X_train, y_train, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({'Feature': features_mi, 'MI_Score': mi_scores}).sort_values('MI_Score', ascending=False)

# So sánh MI vs Spearman
mi_df['Abs_Spearman'] = mi_df['Feature'].map(corr_comparison['Abs_Spearman'])
mi_df['Spearman'] = mi_df['Feature'].map(corr_comparison['Spearman'])

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# MI bar chart
top_mi = mi_df.head(25)
axes[0].barh(range(25), top_mi['MI_Score'], color='steelblue', edgecolor='white')
axes[0].set_yticks(range(25))
axes[0].set_yticklabels(top_mi['Feature'])
axes[0].invert_yaxis()
axes[0].set_xlabel('Mutual Information Score')
axes[0].set_title('Top 25 Features - Mutual Information (phi tuyến)')

# MI vs Spearman scatter
axes[1].scatter(mi_df['Abs_Spearman'], mi_df['MI_Score'], alpha=0.6, s=40)
for _, row in mi_df.head(10).iterrows():
    axes[1].annotate(row['Feature'], (row['Abs_Spearman'], row['MI_Score']),
                     fontsize=7, alpha=0.8)
axes[1].set_xlabel('|Spearman Correlation|')
axes[1].set_ylabel('Mutual Information')
axes[1].set_title('MI vs |Spearman| — Features ở góc trên-trái có quan hệ PHI TUYẾN mạnh')

plt.tight_layout()
plt.show()

print('=== TOP 15 FEATURES THEO MUTUAL INFORMATION ===')
display(mi_df.head(15)[['Feature', 'MI_Score', 'Spearman']].reset_index(drop=True))
print('\n→ MI cao + |Spearman| thấp = quan hệ phi tuyến mạnh → tree-based models sẽ tốt hơn linear.')
print('→ MI cao + |Spearman| cao = quan hệ tuyến tính mạnh → cả linear lẫn tree đều OK.')

---
## 11. Train vs Test Distribution Shift

In [ ]:
# Kiểm tra xem phân phối features có thay đổi giữa train và test không
# Dùng KS-test (Kolmogorov-Smirnov) - H0: cùng phân phối

shift_results = []
for col in CONTINUOUS_COLS:
    ks_stat, ks_p = stats.ks_2samp(train[col].dropna(), test[col].dropna())
    train_mean, test_mean = train[col].mean(), test[col].mean()
    pct_shift = abs(test_mean - train_mean) / (abs(train_mean) + 1e-10) * 100
    shift_results.append({
        'Feature': col,
        'Train_mean': train_mean,
        'Test_mean': test_mean,
        'Mean_shift_%': pct_shift,
        'KS_stat': ks_stat,
        'KS_p': ks_p,
        'Shifted': 'CÓ SHIFT' if ks_p < 0.01 else 'OK'
    })

shift_df = pd.DataFrame(shift_results).sort_values('KS_stat', ascending=False)

print(f'=== DISTRIBUTION SHIFT: TRAIN vs TEST ===')
print(f'Số features bị shift (KS p < 0.01): {(shift_df["Shifted"] == "CÓ SHIFT").sum()} / {len(shift_df)}')
print()
display(shift_df.head(20).reset_index(drop=True))

In [ ]:
# Visual comparison cho top shifted features
top_shifted = shift_df.head(8)['Feature'].tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes_flat = axes.flatten()

for i, col in enumerate(top_shifted):
    ax = axes_flat[i]
    ax.hist(train[col].dropna(), bins=40, alpha=0.5, density=True, label='Train', color='steelblue')
    ax.hist(test[col].dropna(), bins=40, alpha=0.5, density=True, label='Test', color='crimson')
    ks_p = shift_df[shift_df['Feature'] == col]['KS_p'].values[0]
    ax.set_title(f'{col}\nKS p={ks_p:.2e}', fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle('Top 8 Features có Distribution Shift lớn nhất (Train vs Test)', fontsize=14)
plt.tight_layout()
plt.show()
print('→ Features bị shift mạnh có thể gây model performance degradation trên test set.')
print('→ Cần cân nhắc: dùng features ổn định hơn, hoặc dùng mô hình robust với concept drift.')

In [ ]:
# Target shift
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train[TARGET], bins=60, alpha=0.5, density=True, label='Train', color='steelblue')
axes[0].hist(test[TARGET], bins=60, alpha=0.5, density=True, label='Test', color='crimson')
axes[0].set_title('oil_return Distribution: Train vs Test')
axes[0].legend()

# Cumulative distribution
axes[1].hist(train[TARGET], bins=100, alpha=0.5, density=True, cumulative=True, label='Train', color='steelblue')
axes[1].hist(test[TARGET], bins=100, alpha=0.5, density=True, cumulative=True, label='Test', color='crimson')
axes[1].set_title('CDF: Train vs Test')
axes[1].legend()

plt.tight_layout()
plt.show()

ks_stat, ks_p = stats.ks_2samp(train[TARGET], test[TARGET])
print(f'Target KS test: stat={ks_stat:.4f}, p={ks_p:.4f}')
print(f'Train: mean={train[TARGET].mean():.6f}, std={train[TARGET].std():.6f}')
print(f'Test:  mean={test[TARGET].mean():.6f}, std={test[TARGET].std():.6f}')

---
## 12. Tổng kết & Khuyến nghị

In [ ]:
# Tổng hợp EDA findings
print('=' * 80)
print('TỔNG KẾT EDA - 54 FEATURES')
print('=' * 80)

print('\n1. CHẤT LƯỢNG DỮ LIỆU')
print(f'   - Missing values: 0 (sạch)')
print(f'   - Infinite values: 0')
print(f'   - Duplicates: 0')
print(f'   - Dataset: {df.shape[0]} rows x {df.shape[1]} cols')

print('\n2. TARGET (oil_return)')
print(f'   - Phân phối: KHÔNG chuẩn (leptokurtic, fat tails)')
print(f'   - Kurtosis = {train[TARGET].kurtosis():.2f} → đuôi nặng, extreme events phổ biến')
print(f'   - Skewness = {train[TARGET].skew():.2f} → hơi lệch trái')
print(f'   - Stationary: CÓ (ADF + KPSS confirm)')

print('\n3. STATIONARITY')
print(f'   - Biến giá gốc (oil_close, usd_close,...): Non-stationary → KHÔNG dùng trực tiếp')
print(f'   - Biến return/derived: Stationary → OK cho model')

print('\n4. VOLATILITY CLUSTERING')
print(f'   - ARCH effects: CÓ (Ljung-Box p << 0.05 trên squared returns)')
print(f'   - → oil_volatility_7d là feature quan trọng')

print('\n5. CORRELATION VỚI TARGET')
top5_corr = corr_comparison.head(5)
for feat in top5_corr.index:
    print(f'   - {feat}: Spearman = {top5_corr.loc[feat, "Spearman"]:.3f}')
print(f'   → Correlation yếu tổng thể (max |ρ| < 0.3) → cần model phi tuyến')

print('\n6. MULTICOLLINEARITY')
print(f'   - Cặp multicollinear (|ρ| > 0.80): {len(hc_df)} cặp')
if len(danger_vif) > 0:
    print(f'   - Features VIF > 10: {danger_vif["Feature"].tolist()}')
print(f'   → Step 5 reduction đã xử lý bớt, nhưng cần kiểm tra thêm khi chọn feature set')

print('\n7. DISTRIBUTION SHIFT (Train vs Test)')
n_shifted = (shift_df['Shifted'] == 'CÓ SHIFT').sum()
print(f'   - {n_shifted}/{len(shift_df)} features bị shift (KS p < 0.01)')
print(f'   - Top shifted: {shift_df.head(5)["Feature"].tolist()}')
print(f'   → Cần model robust với concept drift')

print('\n8. KHUYẾN NGHỊ CHO MODELING')
print('   a) Dùng dataset_final.csv (33 cột) hoặc tự chọn subset từ 54 cột')
print('   b) Tree-based models (XGBoost, LightGBM) phù hợp hơn linear vì:')
print('      - Correlation yếu & phi tuyến')
print('      - Fat tails trong target')
print('      - Xử lý multicollinearity tự nhiên')
print('   c) Regularization (Ridge/Lasso/ElasticNet) nếu dùng linear models')
print('   d) Time-series cross-validation (không random split)')
print('   e) Cân nhắc loại biến giá gốc (non-stationary + leakage risk)')
print('   f) Monitor feature importance theo thời gian (rolling correlation không ổn định)')

print('\n' + '=' * 80)